## Pengaturan konfigurasi awal

#### 1. Impor modul yang dibutuhkan

In [ ]:
import os
import re
import pandas as pd
from openpyxl import load_workbook

#### 2. Membuat fungsi pendukung

In [ ]:
# Membaca data dari file excel
def read_data_from_excel(file_path, sheet_name):
    # Baca sheet dengan header baris 1-3
    df = pd.read_excel(file_path, sheet_name=sheet_name, header=[0,1,2])

    # Gabungkan header multi-level jadi single-level
    df.columns = [
        "_".join([str(c).strip() for c in col if "Unnamed" not in str(c)])
            .lower()
            .replace(" ", "_")
        for col in df.columns.values
    ]

    # Hapus double underscore kalau ada
    df.columns = [c.replace("__", "_").strip("_") for c in df.columns]

    # Cek hasil kolom
    print(df.columns.tolist())

    # Tampilkan data
    print(df.head())

    return df

# Melakukan cleaning data pada dataframe
def cleaning_data(df):
    # bikin copy biar df asli nggak berubah
    df_clean = df.copy()

    # ambil semua kolom object kecuali 'kabupaten/kota'
    object_cols = [col for col in df_clean.select_dtypes(include=["object"]).columns if col != "kabupaten/kota" and col != "provinsi"]

    # convert semua kolom object itu jadi numeric
    # kalau ada "-", "" atau value non-numeric lain -> otomatis jadi NaN
    for col in object_cols:
        df_clean[col] = pd.to_numeric(df_clean[col], errors="coerce")

    # cek hasil konversi
    print(df_clean.dtypes)

    return df_clean

# Menggabungkan data evaluasi
def concat_evaluasi(list_evaluasi):
    return pd.concat(list_evaluasi, ignore_index=True)    

# Memastikan nama sheet valid
def safe_sheet_name(name):
    return re.sub(r'[:\\/*?\[\]]', "_", str(name))[:31]

# Menulis data ke dalam file excel
def write_to_excel(df, group_col):
    for group_value in df[group_col].unique():
        df_group = df[df[group_col] == group_value]
        kode = str(df_group["kode"].iloc[0])  # ambil kode group_col

        # Nama file sesuai kode group_col
        if group_col == 'provinsi':
            output_file = f"data/output/evaluasi_{kode}.xlsx"
            sheet_name = safe_sheet_name(kode + "00")
        elif group_col == 'kabupaten/kota':
            output_file = f"data/output/evaluasi_{kode[:2]}.xlsx"
            sheet_name = safe_sheet_name(kode)
        

        if os.path.exists(output_file):
            # Load workbook lama
            book = load_workbook(output_file)

            if sheet_name in book.sheetnames:
                # Sheet sudah ada → cari baris terakhir
                ws = book[sheet_name]
                startrow = ws.max_row  # tulis setelah baris terakhir

                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a", if_sheet_exists="overlay") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False, header=False, startrow=startrow)
                print(f"Data {group_value} ditambahkan ke sheet {sheet_name} di {output_file}")

            else:
                # Sheet belum ada → tambahkan sheet baru
                with pd.ExcelWriter(output_file, engine="openpyxl", mode="a") as writer:
                    df_group.to_excel(writer, sheet_name=sheet_name, index=False)
                print(f"Sheet {sheet_name} ditambahkan ke {output_file}")

        else:
            # File belum ada → buat baru
            with pd.ExcelWriter(output_file, engine="openpyxl", mode="w") as writer:
                df_group.to_excel(writer, sheet_name=sheet_name, index=False)
            print(f"File baru dibuat: {output_file}, dengan sheet {sheet_name}")


#### 3. Menentukan lokasi file yang akan digunakan sebagai input data

In [ ]:
file_path = os.path.join("data/input", "Evaluasi.xlsx")

## Evaluasi Provinsi vs Kabupaten/Kota

#### 1. Membaca file Evaluasi.xlsx pada sheet Prov vs KabKot

In [ ]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Prov vs KabKot")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

#### 2. Evaluasi diskrepansi ADHB dan ADHK

In [ ]:
# 1. Ambil kolom yang diawali adhb atau adhk
target_cols = [col for col in df_clean.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat list untuk menampung hasil diskrepansi
evaluasi_diskrepansi = []

# 3. Loop tiap kolom target
for col in target_cols:
    for idx, val in df_clean[col].items():
        if pd.notna(val) and (val < -4.99 or val > 4.99):
            # Pisahkan nama kolom jadi komponen
            # contoh: adhb_pdrb_q1_25 → ['adhb', 'pdrb', 'q1', '25']
            parts = col.split("_")
            
            periode = parts[-1]  # ambil q1 / q2
            record = {
                "kode": df_clean.at[idx, "kode"],
                "provinsi": df_clean.at[idx, "provinsi"],
                "periode": periode,
                "nilai": val,
                "kolom": col,  # tambahan biar tau sumbernya
                "evaluasi": "Diskrepansi lebih dari +- 5 persen"
            }
            evaluasi_diskrepansi.append(record)

# 4. Ubah jadi DataFrame biar enak lihat
evaluasi_diskrepansi = pd.DataFrame(evaluasi_diskrepansi)
print(evaluasi_diskrepansi)

#### 3. Evaluasi distribusi

In [ ]:
# 1. Ambil semua kolom distribusi
dist_cols = [col for col in df_clean.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in dist_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: distribusi_pdrb_q1-25 → q1)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v0 - v1) / v1)
                if selisih > 0.2:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Distribusi lebih dari +- 20 persen"
                    }
                    evaluasi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_dist = pd.DataFrame(evaluasi_dist)
print(evaluasi_dist)

#### 4. Evaluasi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 

In [ ]:
# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y 
growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y")]

# 2. Buat base name tanpa suffix `.1`
pairs = {}
for col in growth_cols:
    base = re.sub(r'\.1$', '', col)  # buang .1 di akhir
    pairs.setdefault(base, []).append(col)

# 3. Evaluasi Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, dan Implisit Y-on-Y
evaluasi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if not c.endswith(".1")][0]
        col1 = [c for c in cols if c.endswith(".1")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Growth berlawanan arah"
                    }
                    evaluasi_growth.append(record)
                elif selisih > 2:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "provinsi": df_clean.at[idx, "provinsi"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Growth lebih dari +- 2 kalinya"
                    }
                    evaluasi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_growth = pd.DataFrame(evaluasi_growth)
print(evaluasi_growth)

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_diskrepansi, evaluasi_dist, evaluasi_growth])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "provinsi")

## Evaluasi Growth dan Revisi

#### 1. Membaca file Evaluasi.xlsx pada sheet Growth & Revisi

In [ ]:
# Membaca data dari file excel
df = read_data_from_excel(file_path, "Growth & Revisi")

# Cleaning data
df_clean = cleaning_data(df)
df_clean.info()

#### 2. Evaluasi revisi ADHB dan ADHK

In [ ]:
# 1. Ambil semua kolom ADHB dan ADHK 
revisi_harga_cols = [col for col in df.columns if col.startswith("adhb") or col.startswith("adhk")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_harga_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi ADHB dan ADHK
evaluasi_revisi_harga = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Angka revisi berlawanan arah"
                    }
                    evaluasi_revisi_harga.append(record)
                elif selisih > 0.1:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Angka revisi lebih dari +- 10 persen"
                    }
                    evaluasi_revisi_harga.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_harga = pd.DataFrame(evaluasi_revisi_harga)
print(evaluasi_revisi_harga)

#### 3. Evaluasi revisi distribusi

In [ ]:
# =OR(ABS((AW4-AV4)/AV4)>0.1, AV4*AW4<0) pkrt 
# =OR(ABS((AZ4-AY4)/AY4)>0.1, AY4*AZ4<0) pklnprt
# =OR(ABS((BC4-BB4)/BB4)>0.1, BB4*BC4<0) pkp
# =OR(ABS((BF4-BE4)/BE4)>0.1, BE4*BF4<0) pmtb
# =BH4*BI4<0 pi
# =BK4*BL4<0 net ekspor 

# 1. Ambil semua kolom distribusi 
revisi_dist_cols = [col for col in df.columns if col.startswith("distribusi")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_dist_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi distribusi
evaluasi_revisi_dist = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df.index:
            v0 = df.at[idx, col0]
            v1 = df.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Distribusi revisi berlawanan arah"
                    }
                    evaluasi_revisi_dist.append(record)
                elif selisih > 0.1 and "pi" not in col1 and "net_ekspor" not in col1:
                    record = {
                        "kode": df.at[idx, "kode"],
                        "kabupaten/kota": df.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Distribusi revisi lebih dari +- 10 persen"
                    }
                    evaluasi_revisi_dist.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_dist = pd.DataFrame(evaluasi_revisi_dist)
print(evaluasi_revisi_dist)

#### 4. Evaluasi revisi growth Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 

In [ ]:
# q-to-q
# =OR(ABS((BO4-BN4)/BN4)>0.5, BN4*BO4<0) pdrb
# =OR(ABS((BR4-BQ4)/BQ4)>0.5, BQ4*BR4<0) pkrt
# =OR(ABS((BU4-BT4)/BT4)>0.5, BT4*BU4<0) pklnprt
# =OR(ABS((BX4-BW4)/BW4)>0.5, BW4*BX4<0) pkp
# =OR(ABS((CA4-BZ4)/BZ4)>0.5, BZ4*CA4<0) pmtb

# y-on-y
# =OR(ABS((CJ4-CI4)/CI4)>0.5, CI4*CJ4<0) pdrb
# =OR(ABS((CM4-CL4)/CL4)>0.5, CL4*CM4<0) pkrt
# =OR(ABS((CP4-CO4)/CO4)>0.5, CO4*CP4<0) pklnprt
# =OR(ABS((CS4-CR4)/CR4)>0.5, CR4*CS4<0) pkp
# =OR(ABS((CV4-CU4)/CU4)>0.5, CU4*CV4<0)pmtb

# c-to-c
# =OR(ABS((DE4-DD4)/DD4)>0.5, DD4*DE4<0) pdrb
# =OR(ABS((DH4-DG4)/DG4)>0.5, DG4*DH4<0) pkrt
# =OR(ABS((DK4-DJ4)/DJ4)>0.5, DJ4*DK4<0) pklnprt
# =OR(ABS((DN4-DM4)/DM4)>0.5, DM4*DN4<0) pkp
# =OR(ABS((DQ4-DP4)/DP4)>0.5, DP4*DQ4<0) pmtb

# implisit q-to-q
# =OR(ABS((DZ4-DY4)/DY4)>0.5, DY4*DZ4<0) pdrb
# =OR(ABS((EC4-EB4)/EB4)>0.5, EB4*EC4<0) pkrt
# =OR(ABS((EF4-EE4)/EE4)>0.5, EE4*EF4<0) pklnprt
# =OR(ABS((EI4-EH4)/EH4)>0.5, EH4*EI4<0) pkp
# =OR(ABS((EL4-EK4)/EK4)>0.5, EK4*EL4<0) pmtb

# implisit y-on-y
# =OR(ABS((EU4-ET4)/ET4)>0.5, ET4*EU4<0) pdrb
# =OR(ABS((EX4-EW4)/EW4)>0.5, EW4*EX4<0) pkrt
# =OR(ABS((FA4-EZ4)/EZ4)>0.5, EZ4*FA4<0) pklnprt
# =OR(ABS((FD4-FC4)/FC4)>0.5, FC4*FD4<0) pkp
# =OR(ABS((FG4-FF4)/FF4)>0.5, FF4*FG4<0) pmtb

# sog q-to-q
# =OR(ABS((FP4-FO4)/FO4)>0.5, FO4*FP4<0) pdrb
# =OR(ABS((FS4-FR4)/FR4)>0.5, FR4*FS4<0) pkrt
# =OR(ABS((FV4-FU4)/FU4)>0.5, FU4*FV4<0) pklnprt
# =OR(ABS((FY4-FX4)/FX4)>0.5, FX4*FY4<0) pkp
# =OR(ABS((GB4-GA4)/GA4)>0.5, GA4*GB4<0) pmtb
# =OR(ABS((GE4-GD4)/GD4)>0.5, GD4*GE4<0) pi
# =OR(ABS((GH4-GG4)/GG4)>0.5, GG4*GH4<0) net ekspor

# sog y-on-y
# =OR(ABS((GK4-GJ4)/GJ4)>0.5, GJ4*GK4<0) pdrb
# =OR(ABS((GN4-GM4)/GM4)>0.5, GM4*GN4<0) pkrt
# =OR(ABS((GQ4-GP4)/GP4)>0.5, GP4*GQ4<0) pklnprt
# =OR(ABS((GT4-GS4)/GS4)>0.5, GS4*GT4<0) pkp
# =OR(ABS((GW4-GV4)/GV4)>0.5, GV4*GW4<0) pmtb
# =OR(ABS((GZ4-GY4)/GY4)>0.5, GY4*GZ4<0) pi
# =OR(ABS((HC4-HB4)/HB4)>0.5, HB4*HC4<0) net ekspor

object_cols = df_clean.select_dtypes(include=["object"]).columns.tolist()
print(object_cols)

# 1. Ambil semua kolom Q-to-Q, Y-on-Y, C-to-C, Implisit Q-to-Q, Implisit Y-on-Y, SOG Q-to-Q, dan SOG Y-on-Y 
revisi_growth_cols = [col for col in df_clean.columns if col.startswith("q-to-q") or col.startswith("y-on-y") or col.startswith("c-to-c") or col.startswith("implisit_q-to-q") or col.startswith("implisit_y-on-y") or col.startswith("sog_q") or col.startswith("sog_y-on-y")]

# 2. Buat base name tanpa suffix `ril/rev`
pairs = {}
for col in revisi_growth_cols:
    if "ril" in col or "rev" in col:
        base = "_".join(col.split("_")[:-1])  # buang suffix ril/rev
        pairs.setdefault(base, []).append(col)

# 3. Evaluasi growth
evaluasi_revisi_growth = []

for base, cols in pairs.items():
    if len(cols) == 2:  # hanya kalau ada pasangan asli + .1
        col0 = [c for c in cols if c.endswith("_ril")][0]
        col1 = [c for c in cols if c.endswith("_rev")][0]

        # Ambil periode (contoh: q-to-q_pdrb_q1-25 → q1-25)
        parts = base.split("_")
        periode = parts[-1]  # ambil q1-25, q2-25, dst

        for idx in df_clean.index:
            v0 = df_clean.at[idx, col0]
            v1 = df_clean.at[idx, col1]

            if pd.notna(v0) and pd.notna(v1) and v1 != 0:
                if type(v0) == str or type(v1) == str:
                    print("ini nilai v0:", v0, "field:", col0, "Type datanya:", type(v0))
                    print("ini nilai v1:", v1, "field:", col1, "Type datanya:", type(v1)) 

                selisih = abs((v1 - v0) / v0)
                if v0 * v1 < 0:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Growth revisi berlawanan arah"
                    }
                    evaluasi_revisi_growth.append(record)
                elif selisih > 0.5:
                    record = {
                        "kode": df_clean.at[idx, "kode"],
                        "kabupaten/kota": df_clean.at[idx, "kabupaten/kota"],
                        "periode": periode,
                        "nilai": v1,
                        "kolom": col0 + " vs " + col1,
                        "evaluasi": "Growth revisi lebih dari +- 50 persen"
                    }
                    evaluasi_revisi_growth.append(record)

# 4. Jadi DataFrame
evaluasi_revisi_growth = pd.DataFrame(evaluasi_revisi_growth)
print(evaluasi_revisi_growth)

## Menggabungkan seluruh hasil evaluasi

In [ ]:
evaluasi = concat_evaluasi([evaluasi_revisi_harga, evaluasi_revisi_dist, evaluasi_revisi_growth])
print(evaluasi.head())

## Menuliskan hasil evaluasi kedalam file excel

In [ ]:
write_to_excel(evaluasi, "kabupaten/kota")